# Adaptive Frames for Hybrid Intelligence
## A simple Minsky-inspired Semantic Data Technologies simulation

**ARU × China Jiliang University — Semantic Data Technologies**

This notebook turns the main ideas in Marvin Minsky's frame theory into a small,
visible and testable pipeline. We will create a movie knowledge graph, compare
competing frames, validate the graph, ask a useful question, repair the graph
with human confirmation and record exactly what changed.

### The idea in one sentence

> A neural or statistical component may **propose** an interpretation; the
> ontology **defines** its meaning; SHACL **checks** the data; a person or agent
> obtains missing evidence; and provenance records every **repair**.

### What you will learn

1. How RDF triples form a knowledge graph.
2. Why an OWL class is only one part of a Minsky-style frame.
3. How to score competing interpretations transparently.
4. How SPARQL asks questions of the graph.
5. How SHACL detects missing or malformed data.
6. How active questions help resolve ambiguity.
7. How to repair data without hiding the original claim.
8. How to export an OWL/RDF graph, reports, tables and diagrams.

## 0. The complete workflow

The notebook follows this loop:

**Observe → retrieve candidate frames → fill slots → validate → ask or act →
repair with provenance → validate again**

The important point is the loop. A validation failure is not the end of the
analysis. It tells us what evidence or representational change may be needed.

In [ ]:
#@title 1. Install the small set of required packages
# Colab already contains pandas, matplotlib and networkx.  We explicitly
# install RDFLib (RDF graphs) and pySHACL (validation) so the notebook also
# works in a new runtime.
!pip -q install "rdflib>=7.0" "pyshacl>=0.30"

In [ ]:
#@title 2. Import libraries and choose consistent colours

# Standard-library tools
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Set
import math
import re

# Data, graph and visualisation libraries
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

# RDFLib represents RDF/OWL graphs in Python.
from rdflib import Graph, Namespace, URIRef, BNode, Literal
from rdflib.namespace import RDF, RDFS, OWL, XSD, SH

# pySHACL validates an RDF data graph against an RDF shapes graph.
from pyshacl import validate

# ARU-inspired colour palette used in all notebook figures.
ARU_NAVY = "#123B5D"
ARU_RED = "#C81D3C"
ARU_GOLD = "#C8A45D"
ARU_BLUE_GREY = "#DCE5EA"
ARU_CREAM = "#F7F2E8"
ARU_PINK = "#F4DDE3"
ARU_GREEN = "#28856B"

plt.rcParams.update({
    "figure.figsize": (10, 5.5),
    "axes.titlesize": 15,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "font.size": 10,
})

print("Libraries loaded. The adaptive-frame simulation is ready.")

## 1. Draw the adaptive-frame architecture

This first diagram is deliberately independent of the ontology. It shows the
logic we will implement in the rest of the notebook.

In [ ]:
#@title 3. Architecture diagram

def draw_workflow():
    '''Draw the six-stage adaptive-frame lifecycle as a directed graph.'''
    stages = [
        ("Observe", "text, tables, RDF"),
        ("Retrieve", "SPARQL + similarity"),
        ("Instantiate", "slots + defaults"),
        ("Validate", "OWL + SHACL + risk"),
        ("Ask / act", "query, tool or human"),
        ("Repair", "provenance + revision"),
    ]

    graph = nx.DiGraph()
    for title, subtitle in stages:
        graph.add_node(title, subtitle=subtitle)
    for index in range(len(stages) - 1):
        graph.add_edge(stages[index][0], stages[index + 1][0])
    graph.add_edge("Repair", "Observe")  # feedback makes it a learning loop

    positions = {
        "Observe": (0, 1), "Retrieve": (1.5, 1), "Instantiate": (3, 1),
        "Validate": (3, 0), "Ask / act": (1.5, 0), "Repair": (0, 0),
    }
    colours = [ARU_CREAM, ARU_BLUE_GREY, ARU_PINK,
               ARU_CREAM, ARU_BLUE_GREY, ARU_PINK]

    fig, ax = plt.subplots(figsize=(12, 5))
    nx.draw_networkx_nodes(
        graph, positions, node_color=colours, node_size=4300,
        edgecolors=ARU_NAVY, linewidths=1.8, ax=ax
    )
    nx.draw_networkx_edges(
        graph, positions, edge_color=ARU_RED, width=2,
        arrows=True, arrowsize=20, connectionstyle="arc3,rad=0.04", ax=ax
    )
    nx.draw_networkx_labels(
        graph, positions,
        labels={node: f"{node}\n{graph.nodes[node]['subtitle']}" for node in graph},
        font_size=9, font_weight="bold", font_color=ARU_NAVY, ax=ax
    )
    ax.set_title("Adaptive-frame lifecycle", color=ARU_NAVY, pad=15)
    ax.axis("off")
    plt.tight_layout()
    plt.savefig("01_adaptive_frame_workflow.png", dpi=180, bbox_inches="tight")
    plt.show()

draw_workflow()

## 2. Create a small movie knowledge graph

An RDF statement is a **triple**:

`subject → predicate → object`

For example:

`TheWalkingDead → hasGenre → Horror`

The example intentionally contains an interpretive problem. The source calls
*The Walking Dead* a `Movie`, but it also provides `seasonCount` and
`episodeCount`, which are strong clues for a television-series frame. We also
leave out its year so that SHACL has something meaningful to detect.

In [ ]:
#@title 4. Build the self-contained demonstration graph

# EX contains terms in our teaching ontology.
EX = Namespace("https://example.org/adaptive-movies#")
PROV = Namespace("http://www.w3.org/ns/prov#")

def build_demo_graph() -> Graph:
    '''Create a compact RDF/OWL graph with both clean and problematic records.'''
    graph = Graph()
    graph.bind("ex", EX)
    graph.bind("owl", OWL)
    graph.bind("rdfs", RDFS)
    graph.bind("prov", PROV)

    # --- Ontology classes ---
    for class_uri, label in [
        (EX.CreativeWork, "Creative work"),
        (EX.Movie, "Movie"),
        (EX.TVSeries, "Television series"),
        (EX.Documentary, "Documentary"),
        (EX.Person, "Person"),
        (EX.Genre, "Genre"),
    ]:
        graph.add((class_uri, RDF.type, OWL.Class))
        graph.add((class_uri, RDFS.label, Literal(label)))

    graph.add((EX.Movie, RDFS.subClassOf, EX.CreativeWork))
    graph.add((EX.TVSeries, RDFS.subClassOf, EX.CreativeWork))
    graph.add((EX.Documentary, RDFS.subClassOf, EX.Movie))

    # --- Properties ---
    datatype_properties = [
        EX.title, EX.releaseYear, EX.startYear, EX.runningMinutes,
        EX.seasonCount, EX.episodeCount, EX.sourceConfidence,
    ]
    object_properties = [EX.hasGenre, EX.directedBy]
    for prop in datatype_properties:
        graph.add((prop, RDF.type, OWL.DatatypeProperty))
    for prop in object_properties:
        graph.add((prop, RDF.type, OWL.ObjectProperty))

    # --- Shared genre and person resources ---
    graph.add((EX.Horror, RDF.type, EX.Genre))
    graph.add((EX.Horror, RDFS.label, Literal("Horror")))
    graph.add((EX.FrankDarabont, RDF.type, EX.Person))
    graph.add((EX.FrankDarabont, RDFS.label, Literal("Frank Darabont")))

    # --- A clean movie record ---
    graph.add((EX.Inception, RDF.type, EX.Movie))
    graph.add((EX.Inception, EX.title, Literal("Inception")))
    graph.add((EX.Inception, EX.releaseYear, Literal("2010", datatype=XSD.gYear)))
    graph.add((EX.Inception, EX.runningMinutes, Literal(148, datatype=XSD.integer)))

    # --- The deliberately ambiguous record ---
    graph.add((EX.TheWalkingDead, RDF.type, EX.Movie))
    graph.add((EX.TheWalkingDead, EX.title, Literal("The Walking Dead")))
    graph.add((EX.TheWalkingDead, EX.hasGenre, EX.Horror))
    graph.add((EX.TheWalkingDead, EX.directedBy, EX.FrankDarabont))
    graph.add((EX.TheWalkingDead, EX.seasonCount, Literal(11, datatype=XSD.integer)))
    graph.add((EX.TheWalkingDead, EX.episodeCount, Literal(177, datatype=XSD.integer)))
    graph.add((EX.TheWalkingDead, EX.sourceConfidence, Literal(0.55, datatype=XSD.decimal)))

    return graph

data_graph = build_demo_graph()
print(f"Created {len(data_graph)} RDF triples.")
print("Example triple:")
print(" ", EX.TheWalkingDead.n3(data_graph.namespace_manager),
      EX.seasonCount.n3(data_graph.namespace_manager),
      Literal(11).n3(data_graph.namespace_manager))

### Optional: inspect your own `movies.owl`

The simulation below remains self-contained. If you want to inspect the larger
course ontology, set `UPLOAD_OWN_ONTOLOGY = True`. The uploaded graph is placed
in `external_graph`; it does not overwrite the controlled experiment.

In [ ]:
#@title 5. Optional OWL upload (leave False for the self-contained simulation)
UPLOAD_OWN_ONTOLOGY = False #@param {type:"boolean"}

external_graph = None
if UPLOAD_OWN_ONTOLOGY:
    try:
        from google.colab import files
        uploaded = files.upload()
        owl_name = next(iter(uploaded))
        external_graph = Graph()
        external_graph.parse(owl_name)  # RDFLib detects RDF/XML or Turtle
        print(f"Loaded {len(external_graph):,} triples from {owl_name}.")
    except Exception as error:
        print("The optional upload was not loaded:", error)
else:
    print("Using the built-in graph. Change the switch to inspect your OWL file.")

In [ ]:
#@title 6. Inspect classes, properties and named resources

def local_name(term) -> str:
    '''Convert a URI into a short readable label for tables and diagrams.'''
    text = str(term)
    return re.split(r"[#/]", text.rstrip("#/"))[-1]

def ontology_summary(graph: Graph) -> pd.DataFrame:
    '''Count common ontology components without assuming a particular namespace.'''
    rows = [
        ("Triples", len(graph)),
        ("OWL classes", len(set(graph.subjects(RDF.type, OWL.Class)))),
        ("Object properties", len(set(graph.subjects(RDF.type, OWL.ObjectProperty)))),
        ("Datatype properties", len(set(graph.subjects(RDF.type, OWL.DatatypeProperty)))),
        ("Typed resources", len(set(graph.subjects(RDF.type, None)))),
    ]
    return pd.DataFrame(rows, columns=["Component", "Count"])

summary_df = ontology_summary(data_graph)
display(summary_df)

In [ ]:
#@title 7. Visualise the example knowledge graph

def rdf_to_networkx(graph: Graph, focus=None) -> nx.DiGraph:
    '''Convert selected RDF triples into a NetworkX graph for teaching.'''
    network = nx.DiGraph()
    hidden_predicates = {RDFS.label, RDFS.subClassOf}

    for subject, predicate, obj in graph:
        # If a focus resource is given, retain triples that directly concern it
        # plus class declarations needed to understand the picture.
        if focus is not None and subject != focus and obj != focus:
            continue
        if predicate in hidden_predicates:
            continue
        s_label = local_name(subject)
        o_label = str(obj) if isinstance(obj, Literal) else local_name(obj)
        network.add_node(s_label, kind="resource")
        network.add_node(o_label, kind="literal" if isinstance(obj, Literal) else "resource")
        network.add_edge(s_label, o_label, label=local_name(predicate))
    return network

def draw_knowledge_graph(graph: Graph, focus=EX.TheWalkingDead, filename="02_initial_graph.png"):
    '''Draw a focused RDF graph with labelled predicates.'''
    network = rdf_to_networkx(graph, focus=focus)
    positions = nx.spring_layout(network, seed=7, k=1.1)
    colours = [ARU_PINK if network.nodes[n]["kind"] == "literal" else ARU_BLUE_GREY
               for n in network.nodes]

    fig, ax = plt.subplots(figsize=(12, 7))
    nx.draw_networkx_nodes(network, positions, node_color=colours, node_size=2600,
                           edgecolors=ARU_NAVY, linewidths=1.4, ax=ax)
    nx.draw_networkx_edges(network, positions, edge_color=ARU_RED, arrows=True,
                           arrowsize=18, width=1.7, ax=ax)
    nx.draw_networkx_labels(network, positions, font_size=8, font_weight="bold", ax=ax)
    nx.draw_networkx_edge_labels(
        network, positions,
        edge_labels=nx.get_edge_attributes(network, "label"),
        font_size=7, font_color=ARU_NAVY, rotate=False, ax=ax
    )
    ax.set_title("Focused RDF graph: The Walking Dead", color=ARU_NAVY)
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(filename, dpi=180, bbox_inches="tight")
    plt.show()

draw_knowledge_graph(data_graph)

## 3. Represent candidate frames

A frame is richer than an OWL class. Here each frame has:

- a semantic class;
- required slots;
- characteristic clues;
- questions that can obtain missing evidence;
- a short explanation.

For readability, the scoring model is intentionally transparent. It is not
presented as a trained scientific classifier.

In [ ]:
#@title 8. Define a small Frame data structure

@dataclass
class Frame:
    '''A compact teaching representation of an adaptive frame.'''
    name: str
    semantic_class: URIRef
    required_slots: Set[URIRef]
    characteristic_slots: Set[URIRef]
    questions: Dict[URIRef, str]
    description: str


frames = [
    Frame(
        name="MovieFrame",
        semantic_class=EX.Movie,
        required_slots={EX.title, EX.releaseYear, EX.runningMinutes},
        characteristic_slots={EX.directedBy, EX.runningMinutes},
        questions={
            EX.releaseYear: "What year was the movie released?",
            EX.runningMinutes: "What is the movie's running time in minutes?",
        },
        description="A primarily single, feature-length audiovisual work.",
    ),
    Frame(
        name="SeriesFrame",
        semantic_class=EX.TVSeries,
        required_slots={EX.title, EX.startYear, EX.seasonCount},
        characteristic_slots={EX.seasonCount, EX.episodeCount},
        questions={
            EX.startYear: "In what year did the television series begin?",
            EX.seasonCount: "How many seasons does the series have?",
        },
        description="An episodic work organised into episodes or seasons.",
    ),
    Frame(
        name="DocumentaryFrame",
        semantic_class=EX.Documentary,
        required_slots={EX.title, EX.releaseYear},
        characteristic_slots={EX.directedBy},
        questions={
            EX.releaseYear: "What year was the documentary released?",
        },
        description="A factual film documenting real subjects or events.",
    ),
]

pd.DataFrame([
    {
        "Frame": frame.name,
        "OWL class": local_name(frame.semantic_class),
        "Required slots": ", ".join(sorted(local_name(p) for p in frame.required_slots)),
        "Characteristic clues": ", ".join(sorted(local_name(p) for p in frame.characteristic_slots)),
    }
    for frame in frames
])

In [ ]:
#@title 9. Score competing frames transparently

def observed_properties(graph: Graph, resource: URIRef) -> Set[URIRef]:
    '''Return all predicates currently used to describe a resource.'''
    return {predicate for predicate in graph.predicates(resource, None)}

def frame_score(graph: Graph, resource: URIRef, frame: Frame) -> dict:
    '''Calculate interpretable components of the teaching score.

    We do not trust the imported rdf:type completely.  Type support is 0.55 for
    the asserted class because the example source reports only 0.55 confidence.
    Strong characteristic properties can therefore favour an alternative frame.
    '''
    properties = observed_properties(graph, resource)
    asserted_types = set(graph.objects(resource, RDF.type))

    source_confidence = next(graph.objects(resource, EX.sourceConfidence), Literal(0.5))
    source_confidence = float(source_confidence)

    # Directly asserted type is evidence, not unquestionable truth.
    type_support = source_confidence if frame.semantic_class in asserted_types else 0.0
    slot_coverage = len(frame.required_slots & properties) / max(len(frame.required_slots), 1)
    cue_coverage = len(frame.characteristic_slots & properties) / max(len(frame.characteristic_slots), 1)
    violation_rate = len(frame.required_slots - properties) / max(len(frame.required_slots), 1)

    # Same components as the paper's pedagogical objective.
    score = (
        0.45 * type_support
        + 0.35 * slot_coverage
        + 0.20 * cue_coverage
        - 0.25 * violation_rate
    )
    return {
        "Frame": frame.name,
        "Type support": round(type_support, 3),
        "Required-slot coverage": round(slot_coverage, 3),
        "Characteristic-cue coverage": round(cue_coverage, 3),
        "Missing-slot penalty": round(violation_rate, 3),
        "Total score": round(score, 3),
    }

scores_before = pd.DataFrame([
    frame_score(data_graph, EX.TheWalkingDead, frame) for frame in frames
]).sort_values("Total score", ascending=False, ignore_index=True)

display(scores_before.round(3))
print("Highest-scoring candidate:", scores_before.iloc[0]["Frame"])

In [ ]:
#@title 10. Plot the score and its components

def plot_frame_scores(scores: pd.DataFrame, filename="03_frame_scores.png"):
    '''Plot total frame scores, retaining the values behind the ranking.'''
    ordered = scores.sort_values("Total score")
    colours = [ARU_GOLD, ARU_BLUE_GREY, ARU_RED]
    ax = ordered.plot.barh(
        x="Frame", y="Total score", legend=False,
        color=colours[:len(ordered)], edgecolor=ARU_NAVY, figsize=(9, 4.5)
    )
    ax.axvline(0, color=ARU_NAVY, linewidth=0.8)
    ax.set_xlabel("Transparent teaching score")
    ax.set_ylabel("")
    ax.set_title("Competing interpretations of The Walking Dead", color=ARU_NAVY)
    for patch, value in zip(ax.patches, ordered["Total score"]):
        ax.text(value + 0.01, patch.get_y() + patch.get_height()/2,
                f"{value:.3f}", va="center", fontsize=9)
    plt.tight_layout()
    plt.savefig(filename, dpi=180, bbox_inches="tight")
    plt.show()

plot_frame_scores(scores_before)

### Interpretation

The ranking does not automatically rewrite the ontology. It says that the
observed season and episode properties make `SeriesFrame` a serious alternative
to the imported `Movie` assertion. We now use a formal query and formal data
validation before proposing a repair.

In [ ]:
#@title 11. Ask a SPARQL competency question

# This query asks: which creative works have episode or season evidence, and
# what type is currently asserted for each one?
query = '''
PREFIX ex:  <https://example.org/adaptive-movies#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?work ?assertedType ?seasons ?episodes
WHERE {
  ?work rdf:type ?assertedType .
  OPTIONAL { ?work ex:seasonCount ?seasons . }
  OPTIONAL { ?work ex:episodeCount ?episodes . }
  FILTER(BOUND(?seasons) || BOUND(?episodes))
}
ORDER BY ?work
'''

query_rows = []
for row in data_graph.query(query):
    query_rows.append({
        "work": local_name(row.work),
        "asserted_type": local_name(row.assertedType),
        "seasons": row.seasons.toPython() if row.seasons else None,
        "episodes": row.episodes.toPython() if row.episodes else None,
    })

sparql_df = pd.DataFrame(query_rows)
display(sparql_df)

## 4. Define SHACL shapes

OWL describes possible meaning and supports inference. SHACL asks whether the
current data meet operational requirements. We define separate shapes for
movies and television series.

Notice the wording: a SHACL violation means **the data do not conform to this
shape**. It does not automatically prove that the real-world statement is false.

In [ ]:
#@title 12. Build Movie and TVSeries shapes

def add_property_shape(shapes: Graph, parent: URIRef, path: URIRef,
                       min_count=1, datatype=None, message="Required value is missing or invalid"):
    '''Attach one reusable property constraint to a SHACL node shape.'''
    rule = BNode()
    shapes.add((parent, SH.property, rule))
    shapes.add((rule, SH.path, path))
    shapes.add((rule, SH.minCount, Literal(min_count)))
    if datatype is not None:
        shapes.add((rule, SH.datatype, datatype))
    shapes.add((rule, SH.message, Literal(message)))


def build_shapes_graph() -> Graph:
    '''Create operational requirements for Movie and TVSeries resources.'''
    shapes = Graph()
    shapes.bind("sh", SH)
    shapes.bind("ex", EX)

    # A Movie requires a title, release year and running time.
    movie_shape = EX.MovieShape
    shapes.add((movie_shape, RDF.type, SH.NodeShape))
    shapes.add((movie_shape, SH.targetClass, EX.Movie))
    add_property_shape(shapes, movie_shape, EX.title, message="Movie needs a title")
    add_property_shape(shapes, movie_shape, EX.releaseYear, datatype=XSD.gYear,
                       message="Movie needs one release year (xsd:gYear)")
    add_property_shape(shapes, movie_shape, EX.runningMinutes, datatype=XSD.integer,
                       message="Movie needs an integer running time")

    # A television series requires a title, start year and season count.
    series_shape = EX.TVSeriesShape
    shapes.add((series_shape, RDF.type, SH.NodeShape))
    shapes.add((series_shape, SH.targetClass, EX.TVSeries))
    add_property_shape(shapes, series_shape, EX.title, message="Series needs a title")
    add_property_shape(shapes, series_shape, EX.startYear, datatype=XSD.gYear,
                       message="Series needs one start year (xsd:gYear)")
    add_property_shape(shapes, series_shape, EX.seasonCount, datatype=XSD.integer,
                       message="Series needs an integer season count")
    return shapes

shapes_graph = build_shapes_graph()
print(f"Created {len(shapes_graph)} SHACL triples.")

In [ ]:
#@title 13. Validate and turn the SHACL report into a readable table

def validation_table(graph: Graph, shapes: Graph):
    '''Run SHACL and return conformance, report graph, text and a tidy table.'''
    conforms, report_graph, report_text = validate(
        data_graph=graph,
        shacl_graph=shapes,
        inference="rdfs",
        abort_on_first=False,
        allow_infos=True,
        allow_warnings=True,
    )

    rows = []
    for result in report_graph.subjects(RDF.type, SH.ValidationResult):
        focus = report_graph.value(result, SH.focusNode)
        path = report_graph.value(result, SH.resultPath)
        message = report_graph.value(result, SH.resultMessage)
        severity = report_graph.value(result, SH.resultSeverity)
        rows.append({
            "Focus node": local_name(focus) if focus else "",
            "Property": local_name(path) if path else "node-level",
            "Severity": local_name(severity) if severity else "Violation",
            "Message": str(message) if message else "",
        })
    return conforms, report_graph, report_text, pd.DataFrame(rows)


conforms_before, report_before, report_text_before, issues_before = validation_table(
    data_graph, shapes_graph
)

print("Does the graph conform?", conforms_before)
display(issues_before if not issues_before.empty else pd.DataFrame({"Result": ["No issues"]}))

In [ ]:
#@title 14. Visual quality dashboard

def plot_issue_dashboard(issues: pd.DataFrame, title: str, filename: str):
    '''Show how many violations affect each property.'''
    if issues.empty:
        counts = pd.Series({"No violations": 0})
    else:
        counts = issues["Property"].value_counts().sort_values()

    fig, ax = plt.subplots(figsize=(9, 4.5))
    bars = ax.barh(counts.index, counts.values, color=ARU_RED, edgecolor=ARU_NAVY)
    ax.set_xlabel("Number of SHACL results")
    ax.set_title(title, color=ARU_NAVY)
    ax.set_xlim(0, max(1, int(counts.max()) + 1))
    for bar, value in zip(bars, counts.values):
        ax.text(value + 0.05, bar.get_y() + bar.get_height()/2,
                str(int(value)), va="center")
    plt.tight_layout()
    plt.savefig(filename, dpi=180, bbox_inches="tight")
    plt.show()

plot_issue_dashboard(issues_before, "Data quality before frame repair",
                     "04_quality_before.png")

## 5. Ask the most useful missing question

Minsky proposed that empty terminals behave like questions. We implement a
simple version of expected information gain:

`question value = discrimination × relevance ÷ cost`

The values below are deliberately visible and editable. A research system would
estimate them from data and validate their calibration.

In [ ]:
#@title 15. Rank active questions

def candidate_questions(graph: Graph, resource: URIRef, candidates: List[Frame]) -> pd.DataFrame:
    '''Rank questions for slots missing from one or more candidate frames.'''
    present = observed_properties(graph, resource)

    # Discrimination describes how strongly a value separates candidate frames.
    # Cost represents time, money or user effort on a 0--1 scale.
    metadata = {
        EX.releaseYear: (0.55, 0.20),
        EX.runningMinutes: (0.90, 0.25),
        EX.startYear: (0.80, 0.20),
        EX.seasonCount: (0.95, 0.25),
    }

    rows = []
    seen = set()
    for frame in candidates:
        for slot, question in frame.questions.items():
            if slot in present or slot in seen:
                continue
            seen.add(slot)
            discrimination, cost = metadata.get(slot, (0.5, 0.5))
            relevance = 1.0 if slot in frame.required_slots else 0.6
            value = discrimination * relevance / (cost + 0.05)
            rows.append({
                "Slot": local_name(slot),
                "Question": question,
                "Discrimination": discrimination,
                "Cost": cost,
                "Question value": round(value, 3),
            })
    return pd.DataFrame(rows).sort_values("Question value", ascending=False, ignore_index=True)


# Use the two highest-scoring frame hypotheses.
top_names = set(scores_before.head(2)["Frame"])
top_frames = [frame for frame in frames if frame.name in top_names]
questions_df = candidate_questions(data_graph, EX.TheWalkingDead, top_frames)
display(questions_df)
print("Recommended next question:", questions_df.iloc[0]["Question"])

## 6. Apply a transparent, human-confirmed repair

Assume that an approved source confirms:

- the work is a television series;
- it began in 2010;
- the existing season and episode counts are valid.

The notebook uses a Boolean switch to represent human confirmation. In a real
application this should be an authenticated approval event, not a notebook
variable.

In [ ]:
#@title 16. Confirm and apply the semantic repair
HUMAN_CONFIRMS_RECLASSIFICATION = True #@param {type:"boolean"}

# Keep a complete copy so the original state can always be reconstructed.
original_graph = Graph()
for triple in data_graph:
    original_graph.add(triple)
for prefix, namespace in data_graph.namespaces():
    original_graph.bind(prefix, namespace)

audit_graph = Graph()
audit_graph.bind("ex", EX)
audit_graph.bind("prov", PROV)

if HUMAN_CONFIRMS_RECLASSIFICATION:
    old_triple = (EX.TheWalkingDead, RDF.type, EX.Movie)
    new_triple = (EX.TheWalkingDead, RDF.type, EX.TVSeries)

    # Change the working graph only after confirmation.
    data_graph.remove(old_triple)
    data_graph.add(new_triple)
    data_graph.add((EX.TheWalkingDead, EX.startYear,
                    Literal("2010", datatype=XSD.gYear)))

    # Represent the change as a provenance activity.
    activity = EX.Reclassification_2026_10_18
    old_statement = EX.OriginalMovieAssertion
    new_statement = EX.RevisedSeriesAssertion
    approver = EX.CourseDemonstrator

    audit_graph.add((activity, RDF.type, PROV.Activity))
    audit_graph.add((activity, RDFS.label, Literal("Human-confirmed frame repair")))
    audit_graph.add((activity, PROV.wasAssociatedWith, approver))
    audit_graph.add((activity, PROV.used, old_statement))
    audit_graph.add((activity, PROV.generated, new_statement))
    audit_graph.add((activity, PROV.endedAtTime,
                     Literal(datetime.now(timezone.utc).isoformat(), datatype=XSD.dateTime)))

    audit_graph.add((approver, RDF.type, PROV.Agent))
    audit_graph.add((old_statement, RDF.type, PROV.Entity))
    audit_graph.add((old_statement, RDF.subject, EX.TheWalkingDead))
    audit_graph.add((old_statement, RDF.predicate, RDF.type))
    audit_graph.add((old_statement, RDF.object, EX.Movie))
    audit_graph.add((new_statement, RDF.type, PROV.Entity))
    audit_graph.add((new_statement, RDF.subject, EX.TheWalkingDead))
    audit_graph.add((new_statement, RDF.predicate, RDF.type))
    audit_graph.add((new_statement, RDF.object, EX.TVSeries))
    audit_graph.add((new_statement, PROV.wasRevisionOf, old_statement))
    audit_graph.add((new_statement, PROV.wasGeneratedBy, activity))

    print("Repair applied and provenance recorded.")
else:
    print("No semantic change was made. The system abstains pending confirmation.")

In [ ]:
#@title 17. Revalidate the repaired graph

conforms_after, report_after, report_text_after, issues_after = validation_table(
    data_graph, shapes_graph
)

print("Does the repaired graph conform?", conforms_after)
display(issues_after if not issues_after.empty else pd.DataFrame({"Result": ["No SHACL violations"]}))

# Re-score the candidate frames after adding verified evidence.
scores_after = pd.DataFrame([
    frame_score(data_graph, EX.TheWalkingDead, frame) for frame in frames
]).sort_values("Total score", ascending=False, ignore_index=True)
display(scores_after.round(3))

In [ ]:
#@title 18. Compare quality before and after repair

before_count = len(issues_before)
after_count = len(issues_after)

comparison_df = pd.DataFrame({
    "State": ["Before repair", "After repair"],
    "SHACL results": [before_count, after_count],
})
display(comparison_df)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
bars = ax.bar(comparison_df["State"], comparison_df["SHACL results"],
              color=[ARU_RED, ARU_GREEN], edgecolor=ARU_NAVY, width=0.58)
ax.set_ylabel("Number of SHACL results")
ax.set_title("Validation before and after human-confirmed repair", color=ARU_NAVY)
ax.set_ylim(0, max(1, before_count + 1))
for bar, value in zip(bars, comparison_df["SHACL results"]):
    ax.text(bar.get_x() + bar.get_width()/2, value + 0.05,
            str(value), ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig("05_validation_before_after.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
#@title 19. Inspect the provenance trail in plain language

provenance_query = '''
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?activity ?label ?agent ?generated ?time
WHERE {
  ?activity a prov:Activity ;
            rdfs:label ?label ;
            prov:wasAssociatedWith ?agent ;
            prov:generated ?generated ;
            prov:endedAtTime ?time .
}
'''

provenance_rows = []
for row in audit_graph.query(provenance_query):
    provenance_rows.append({
        "Activity": local_name(row.activity),
        "Description": str(row.label),
        "Agent": local_name(row.agent),
        "Generated": local_name(row.generated),
        "Time": str(row.time),
    })

display(pd.DataFrame(provenance_rows))

In [ ]:
#@title 20. Draw the repaired graph
draw_knowledge_graph(data_graph, filename="06_repaired_graph.png")

## 7. Export reproducible artefacts

We export the original graph, repaired graph, SHACL shapes, validation reports,
provenance trail and CSV tables. This separation is essential: another person
can reconstruct both what was received and what was changed.

In [ ]:
#@title 21. Save graphs, reports, tables and a compact manifest

output_dir = Path("adaptive_frame_outputs")
output_dir.mkdir(exist_ok=True)

# Graphs in Turtle and RDF/XML (OWL-compatible) formats.
original_graph.serialize(output_dir / "original_graph.ttl", format="turtle")
data_graph.serialize(output_dir / "repaired_graph.ttl", format="turtle")
data_graph.serialize(output_dir / "repaired_graph.owl", format="xml")
shapes_graph.serialize(output_dir / "shapes_graph.ttl", format="turtle")
audit_graph.serialize(output_dir / "repair_provenance.ttl", format="turtle")
report_before.serialize(output_dir / "validation_before.ttl", format="turtle")
report_after.serialize(output_dir / "validation_after.ttl", format="turtle")

# Human-readable tables.
scores_before.to_csv(output_dir / "frame_scores_before.csv", index=False)
scores_after.to_csv(output_dir / "frame_scores_after.csv", index=False)
issues_before.to_csv(output_dir / "issues_before.csv", index=False)
issues_after.to_csv(output_dir / "issues_after.csv", index=False)
questions_df.to_csv(output_dir / "active_questions.csv", index=False)
sparql_df.to_csv(output_dir / "sparql_results.csv", index=False)

# A small text manifest helps users understand the folder without opening code.
manifest = f'''Adaptive-frame simulation outputs
Generated: {datetime.now(timezone.utc).isoformat()}
Original triples: {len(original_graph)}
Repaired triples: {len(data_graph)}
SHACL results before: {len(issues_before)}
SHACL results after: {len(issues_after)}
Human confirmation: {HUMAN_CONFIRMS_RECLASSIFICATION}
'''
(output_dir / "MANIFEST.txt").write_text(manifest, encoding="utf-8")

print("Saved:")
for path in sorted(output_dir.iterdir()):
    print(" -", path)

In [ ]:
#@title 22. Optional: create one downloadable ZIP archive

import shutil
archive_path = shutil.make_archive("adaptive_frame_outputs", "zip", output_dir)
print("Created", archive_path)

try:
    from google.colab import files
    # Uncomment the next line when you want Colab to start the download.
    # files.download(archive_path)
    print("In Colab, remove the # before files.download(...) to download the ZIP.")
except ImportError:
    print("The ZIP is available in the current Jupyter working directory.")

## 8. What the simulation demonstrates—and what it does not

### Demonstrated

- one entity can activate competing frames;
- imported `rdf:type` is evidence rather than unquestionable truth;
- SPARQL retrieves semantically structured evidence;
- SHACL reports operational incompleteness;
- a missing slot can generate an active question;
- semantic repair can require human confirmation;
- provenance can preserve the original and revised claims;
- quality can be compared before and after repair.

### Not yet demonstrated

- a trained neural retriever;
- calibrated probabilities;
- multimodal grounding from images or video;
- causal inference;
- production access control;
- large-scale ontology evolution;
- clinical or other high-stakes validation.

These limitations are deliberate. The notebook is a transparent teaching model
and a starting point for research, not a production decision system.

## 9. Student extension challenges

1. **Add a `ShortFilmFrame`.** Decide which slots are required and how duration
   should affect the score.
2. **Create a contradiction.** Add two different release years from two sources.
   Represent both rather than deleting one.
3. **Add a warning shape.** Treat an unusually large running time as a warning,
   not a hard violation.
4. **Use the course `movies.owl`.** Map its namespace and properties into the
   frame definitions.
5. **Add source reliability.** Compare an official database with an anonymous
   text extraction.
6. **Add a repair approval form.** Store the approver and a written rationale.
7. **Run an ablation.** Remove SHACL or remove characteristic clues and measure
   what changes.
8. **Build a clinical version.** Use synthetic data only and require explicit
   human confirmation before any eligibility conclusion.

### Final reflection

Minsky's lasting insight is not simply that knowledge has slots. It is that an
intelligent representation should guide attention, expose failed expectations,
ask useful questions and learn from repair. Semantic Data Technologies provide
the formal substrate; modern machine learning provides flexible proposal and
retrieval; provenance and governance make the cycle accountable.